# Лабораторная работа №14: Генерация изображений (Stable Diffusion, ControlNet)

**Студент:** [ФИО]  
**Группа:** [Номер группы]

## Цель работы
Изучить основы генерации изображений с помощью диффузионных моделей:
- Работа со Stable Diffusion через библиотеку diffusers
- Использование ControlNet для контроля композиции
- Применение LoRA для стилизации
- Text-to-Image и Image-to-Image генерация

## Теоретические сведения

### Диффузионные модели
Диффузионные модели генерируют изображения путем постепенного удаления шума из случайного тензора. Процесс состоит из двух фаз:
1. **Forward diffusion** — постепенное добавление шума к изображению
2. **Reverse diffusion** — обучение модели восстанавливать изображение из шума

### Stable Diffusion
Stable Diffusion — латентная диффузионная модель, работающая в сжатом пространстве:
- VAE (Variational Autoencoder) — сжатие/восстановление изображений
- U-Net — предсказание шума в латентном пространстве
- CLIP Text Encoder — кодирование текстовых промптов

### ControlNet
ControlNet — архитектура для дополнительного контроля над генерацией:
- Карты глубины (depth maps)
- Карты кромок (canny edges)
- Позы людей (pose estimation)
- Сегментация (semantic segmentation)

### LoRA (Low-Rank Adaptation)
LoRA — эффективный метод дообучения:
- Замораживание основных весов модели
- Обучение небольших адаптерных матриц
- Быстрая загрузка и переключение стилей

## Задание

### Базовый уровень
1. Установите необходимые библиотеки
2. Загрузите предобученную модель Stable Diffusion
3. Сгенерируйте изображения по текстовому описанию
4. Используйте negative prompts для улучшения качества

### Продвинутый уровень
1. Реализуйте Image-to-Image генерацию
2. Примените ControlNet для контроля позы/границ
3. Интегрируйте LoRA для стилизации

### Индивидуальный уровень
1. Создайте пайплайн для генерации последовательности кадров
2. Исследуйте влияние различных параметров на качество
3. Оптимизируйте для работы на GPU с ограниченной памятью

## Ход работы

In [ ]:
# Установка необходимых библиотек
!pip install -q diffusers transformers accelerate safetensors torch torchvision pillow controlnet-aux

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, ControlNetModel, UniPCMultistepScheduler
from PIL import Image
import numpy as np
import cv2
from controlnet_aux import CannyDetector
import warnings
warnings.filterwarnings('ignore')

# Проверка доступности GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")
if device == "cpu":
    print("Внимание: Для быстрой генерации рекомендуется использовать GPU (Google Colab с GPU)"

In [ ]:
# Шаг 1: Загрузка базовой модели Stable Diffusion
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_safetensors=True
)

pipe = pipe.to(device)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# Оптимизация памяти для GPU
if device == "cuda":
    pipe.enable_attention_slicing()
    # pipe.enable_xformers_memory_efficient_attention()  # Раскомментировать если доступен xformers

print("Модель загружена")

In [ ]:
# Шаг 2: Text-to-Image генерация
prompt = "a beautiful sunset over mountains, highly detailed, 8k, photorealistic"
negative_prompt = "ugly, blurry, low quality, distorted, deformed"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=25,
    guidance_scale=7.5,
    width=512,
    height=512,
    generator=torch.manual_seed(42)
).images[0]

display(image)

In [ ]:
# Шаг 3: Сравнение разных промптов
prompts = [
    "cyberpunk city at night, neon lights, rain",
    "cute cat sitting in a garden, flowers, sunlight",
    "astronaut riding a horse on Mars, realistic, cinematic"
]

images = pipe(
    prompts,
    negative_prompt=negative_prompt * len(prompts),
    num_inference_steps=20,
    guidance_scale=7.5,
    width=512,
    height=512
).images

# Отображение результатов
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, prompt in zip(axes, images, prompts):
    ax.imshow(img)
    ax.set_title(prompt[:40] + "...", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Шаг 4: Image-to-Image генерация
img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_safetensors=True
)
img2img_pipe = img2img_pipe.to(device)
img2img_pipe.scheduler = UniPCMultistepScheduler.from_config(img2img_pipe.scheduler.config)

# Загрузка исходного изображения
init_image = image.resize((512, 512))

new_prompt = "winter landscape, snow covered mountains, cold atmosphere"

img2img_output = img2img_pipe(
    prompt=new_prompt,
    image=init_image,
    strength=0.75,  # Степень изменения (0-1)
    guidance_scale=7.5,
    num_inference_steps=25
).images[0]

# Сравнение
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(init_image)
axes[0].set_title("Исходное изображение")
axes[0].axis('off')
axes[1].imshow(img2img_output)
axes[1].set_title("После Image-to-Image")
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Шаг 5: ControlNet для контроля границ (Canny)
# Загрузка ControlNet модели
controlnet_canny = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_canny",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_safetensors=True
)
controlnet_canny = controlnet_canny.to(device)

control_pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    controlnet=controlnet_canny,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_safetensors=True
)
control_pipe = control_pipe.to(device)
control_pipe.scheduler = UniPCMultistepScheduler.from_config(control_pipe.scheduler.config)

print("ControlNet загружен")

In [ ]:
# Детекция границ Canny
def apply_canny(image, low_threshold=100, high_threshold=200):
    image_np = np.array(image)
    image_gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(image_gray, low_threshold, high_threshold)
    return Image.fromarray(edges).convert("RGB")

# Создание карты границ
canny_image = apply_canny(img2img_output)

# Отображение
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img2img_output)
axes[0].set_title("Исходное изображение")
axes[0].axis('off')
axes[1].imshow(canny_image)
axes[1].set_title("Canny edges")
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Генерация с использованием ControlNet
control_prompt = "fantasy castle on a hill, magical atmosphere, fantasy art"

control_output = control_pipe(
    prompt=control_prompt,
    image=canny_image,
    negative_prompt=negative_prompt,
    num_inference_steps=25,
    guidance_scale=7.5
).images[0]

# Сравнение
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(canny_image)
axes[0].set_title("Canny edges (контроль)")
axes[0].axis('off')
axes[1].imshow(img2img_output)
axes[1].set_title("Оригинал")
axes[1].axis('off')
axes[2].imshow(control_output)
axes[2].set_title("Сгенерировано с ControlNet")
axes[2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Шаг 6: Работа с LoRA (пример загрузки)
# Примечание: В реальном проекте замените на актуальный LoRA checkpoint

from diffusers import StableDiffusionPipeline

# Пример загрузки LoRA (закомментировано, так как требует конкретного checkpoint)
# pipe.load_lora_weights("path/to/lora/weights", weight_name="pytorch_lora_weights.safetensors")

print("LoRA интеграция:")
print("1. Найдите LoRA веса на Hugging Face Hub (например, стили аниме, реализм и т.д.)")
print("2. Загрузите с помощью pipe.load_lora_weights()")
print("3. Используйте pipe.unload_lora_weights() для сброса")
print("\nПример кода:")
print('''
# Загрузка LoRA
pipe.load_lora_weights("nerijs/pixel-art-xl", weight_name="pixel-art-xl.safetensors")

# Генерация в стиле LoRA
image = pipe(prompt="a cute cat", generator=torch.manual_seed(42)).images[0]

# Выгрузка LoRA
pipe.unload_lora_weights()
''')

In [ ]:
# Шаг 7: Эксперименты с параметрами
def compare_parameters(prompt, seeds=[42, 123, 456], guidance_scales=[5, 7.5, 10]):
    fig, axes = plt.subplots(len(seeds), len(guidance_scales), figsize=(15, 15))
    
    if len(seeds) == 1:
        axes = [axes]
    if len(guidance_scales) == 1:
        axes = [[ax] for ax in axes]
    
    for i, seed in enumerate(seeds):
        for j, scale in enumerate(guidance_scales):
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=20,
                guidance_scale=scale,
                generator=torch.manual_seed(seed)
            ).images[0]
            
            axes[i][j].imshow(result)
            axes[i][j].set_title(f"Seed={seed}, CFG={scale}")
            axes[i][j].axis('off')
    
    plt.tight_layout()
    plt.show()

test_prompt = "portrait of a warrior, detailed armor, dramatic lighting"
compare_parameters(test_prompt)

## Контрольные вопросы

1. Как работает процесс диффузии и чем он отличается от GAN?
2. Зачем используется латентное пространство в Stable Diffusion?
3. Как параметр guidance_scale влияет на результат генерации?
4. Какие типы ControlNet вы знаете и для каких задач они подходят?
5. В чем преимущества LoRA перед полным fine-tuning модели?

## Выводы

В данной лабораторной работе были изучены основы генерации изображений:
- Освоена работа со Stable Diffusion для text-to-image и image-to-image
- Реализован контроль композиции через ControlNet (Canny edges)
- Изучены принципы работы LoRA для стилизации
- Проанализировано влияние параметров генерации на качество результата

Диффузионные модели открывают широкие возможности для создания контента, а дополнительные инструменты контроля позволяют точно управлять результатом генерации.